# MoE Compression Baselines

Models: Qwen1.5-MoE-A2.7B-Chat, tiny-random-Mixtral, Phi-mini-MoE-instruct  
Calibration: MMLU train, GSM8k train  
Evaluation: MMLU test (all), GSM8k test  
Methods: REAP pruning, HC-SMoE merging

**Workflow**: run baseline eval → copy compression commands to Docker terminal → run post-compression eval

In [1]:
%env CUDA_VISIBLE_DEVICES=6

import glob
import os
import re
import sys

import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Shared QA-eval module lives in MoE_Optimization/common/
sys.path.insert(0, os.path.abspath("./MoE_Optimization"))
from common.qa_eval import evaluate_qa

env: CUDA_VISIBLE_DEVICES=6


/home/anmilka/miniconda3/envs/common/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
MODELS = [
    "Qwen/Qwen1.5-MoE-A2.7B-Chat",
    "-",
    "microsoft/Phi-mini-MoE-instruct",
]

# Calibration datasets for pruning / merging.
# The MMLU slice matches the fusion-experiment setup:
#   auxiliary_train split, 1024 batches  (Qwen_MoE_experts_fusion_experiment TRAIN_SIZE = 1024)
CALIB_DATASETS = {
    "mmlu_train":           "cais/mmlu[all](auxiliary_train):128",
    "commonsense_qa_train": "tau/commonsense_qa(train):128",
    "openbookqa_train":     "allenai/openbookqa[main](train):128",
    "winogrande_train":     "allenai/winogrande[winogrande_debiased](train):128",
}

COMPRESSION_RATIOS = [0.06, 0.12, 0.19]
SEED = 42
DEVICES = "0,1"
BATCH_SIZE = 8

# ── Evaluation config ────────────────────────────────────────────────────────
# All benchmarks: 1 000 random examples, seed 42.
MMLU_EVAL_KWARGS       = {"path": "cais/mmlu",          "name": "all",                  "split": "test"}
MMLU_EVAL_N            = 1000

COMMONSENSE_EVAL_KWARGS = {"path": "tau/commonsense_qa",                                "split": "validation"}
COMMONSENSE_EVAL_N     = 1000

OPENBOOKQA_EVAL_KWARGS = {"path": "allenai/openbookqa", "name": "main",                 "split": "test"}
OPENBOOKQA_EVAL_N      = 1000

WINOGRANDE_EVAL_KWARGS = {"path": "allenai/winogrande", "name": "winogrande_debiased",  "split": "validation"}
WINOGRANDE_EVAL_N      = 1000

results = []

## Helpers

In [3]:
def _sanitize(s: str) -> str:
    """Same as str_to_directory_name in main.py."""
    return re.sub(r"[^\w\-_.]", "_", s)


def _dataset_dir(calib_spec: str) -> str:
    """Convert composite spec to directory name (same logic as create_results_directory)."""
    return _sanitize(calib_spec.split("/")[-1])


def _model_dir(model_name: str) -> str:
    return _sanitize(model_name.split("/")[-1])


def pruned_model_path(model_name, calib_spec, ratio):
    """Predict the pruned model directory. Uses glob to handle renorm_true/false."""
    base = f"artifacts/{_model_dir(model_name)}/{_dataset_dir(calib_spec)}/pruned_models"
    pattern = f"{base}/reap*-seed_{SEED}-{ratio:.2f}"
    matches = sorted(glob.glob(pattern))
    if matches:
        return matches[0]
    return f"{base}/reap-seed_{SEED}-{ratio:.2f}"  # fallback


def merged_model_path(model_name, calib_spec, ratio):
    """Predict the merged model directory (hc_smoe.sh logic)."""
    base = f"artifacts/{_model_dir(model_name)}/{_dataset_dir(calib_spec)}/merged_models"
    return f"{base}/hc_smoe-seed_{SEED}_{ratio}/hc_smoe"


def load_model(model_path, base_model_name=None):
    """Load model + tokenizer.

    Some pruned/merged models are saved with a custom config class (e.g. PhiMoEConfig
    from configuration_slimmoe.py) that AutoTokenizer cannot map to a tokenizer class.
    In that case we fall back to the original base model — the tokenizer never changes
    during pruning or merging.
    """
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    except ValueError:
        if base_model_name is None:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
            base_model_name = getattr(cfg, "_name_or_path", None) or getattr(cfg, "name_or_path", None)
        if not base_model_name:
            raise
        print(f"    [load_model] custom config — loading tokenizer from base: {base_model_name}")
        tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_path, device_map="auto", dtype="auto", trust_remote_code=True,
    )
    model.eval()
    return model, tokenizer


# ── Evaluation wrappers ──────────────────────────────────────────────────────
# Both call the shared evaluate_qa from MoE_Optimization/common/qa_eval.py.
# MMLU: 1 000 random test examples (matches fusion-experiment VALIDATION_SIZE).
# CommonsenseQA: full validation split (1 221 examples).

def eval_mmlu(model, tokenizer):
    acc = evaluate_qa(
        model, tokenizer,
        dataset_kwargs=MMLU_EVAL_KWARGS,
        n=MMLU_EVAL_N,
        seed=SEED,
        batch_size=BATCH_SIZE,
    )
    print(f"    MMLU ({MMLU_EVAL_N} samples): {acc:.4f}")
    return acc


def eval_commonsense(model, tokenizer):
    acc = evaluate_qa(
        model, tokenizer,
        dataset_kwargs=COMMONSENSE_EVAL_KWARGS,
        n=COMMONSENSE_EVAL_N,
        seed=SEED,
        batch_size=BATCH_SIZE,
    )
    print(f"    CommonsenseQA ({COMMONSENSE_EVAL_N} samples): {acc:.4f}")
    return acc


def eval_openbookqa(model, tokenizer):
    acc = evaluate_qa(
        model, tokenizer,
        dataset_kwargs=OPENBOOKQA_EVAL_KWARGS,
        n=OPENBOOKQA_EVAL_N,
        seed=SEED,
        batch_size=BATCH_SIZE,
    )
    print(f"    OpenBookQA ({OPENBOOKQA_EVAL_N} samples): {acc:.4f}")
    return acc


def eval_winogrande(model, tokenizer):
    acc = evaluate_qa(
        model, tokenizer,
        dataset_kwargs=WINOGRANDE_EVAL_KWARGS,
        n=WINOGRANDE_EVAL_N,
        seed=SEED,
        batch_size=BATCH_SIZE,
    )
    print(f"    Winogrande ({WINOGRANDE_EVAL_N} samples): {acc:.4f}")
    return acc


def evaluate_all(model, tokenizer, label, model_name):
    """Run all 4 benchmarks, print every metric, return rows."""
    print(f"\n  === {label} ===")
    rows = []

    mmlu_acc = eval_mmlu(model, tokenizer)
    rows.append({"model": model_name, "variant": label, "benchmark": "MMLU", "accuracy": mmlu_acc})

    csqa_acc = eval_commonsense(model, tokenizer)
    rows.append({"model": model_name, "variant": label, "benchmark": "CommonsenseQA", "accuracy": csqa_acc})

    obqa_acc = eval_openbookqa(model, tokenizer)
    rows.append({"model": model_name, "variant": label, "benchmark": "OpenBookQA", "accuracy": obqa_acc})

    wino_acc = eval_winogrande(model, tokenizer)
    rows.append({"model": model_name, "variant": label, "benchmark": "Winogrande", "accuracy": wino_acc})

    print(f"\n  --- Summary ({label}) ---")
    for r in rows:
        print(f"    {r['benchmark']:20s} {r['accuracy']:.4f}")
    print()
    return rows


def print_reap_cmds(model_name):
    """Print REAP pruning commands for Docker terminal."""
    print("# REAP pruning — run in Docker terminal:")
    for calib_name, calib_spec in CALIB_DATASETS.items():
        for ratio in COMPRESSION_RATIOS:
            print(f"\n# {calib_name}, ratio={ratio}")
            print(
                f'CUDA_VISIBLE_DEVICES={DEVICES} '
                f'bash experiments/pruning-cli.sh '
                f'{DEVICES} {model_name} reap {SEED} {ratio} '
                f'"{calib_spec}" false false false false false'
            )


def print_hc_smoe_cmds(model_name):
    """Print HC-SMoE merging commands for Docker terminal."""
    print("# HC-SMoE merging — run in Docker terminal:")
    for calib_name, calib_spec in CALIB_DATASETS.items():
        for ratio in COMPRESSION_RATIOS:
            print(f"\n# {calib_name}, ratio={ratio}")
            print(
                f'CUDA_VISIBLE_DEVICES={DEVICES} '
                f'bash experiments/merging-cli.sh '
                f'{DEVICES} {model_name} hc_smoe {SEED} {ratio} '
                f'"{calib_spec}" false false false false false'
            )


def eval_pruned(model_name):
    """Find and evaluate all pruned models for a given base model."""
    rows = []
    for calib_name, calib_spec in CALIB_DATASETS.items():
        for ratio in COMPRESSION_RATIOS:
            path = pruned_model_path(model_name, calib_spec, ratio)
            label = f"reap|{calib_name}|r={ratio}"
            print(f"\n--- {label} ---")
            print(f"    Path: {path}")
            if not glob.glob(f"{path}/*.safetensors") and not glob.glob(f"{path}/*.bin"):
                print(f"    SKIP: no model files found")
                continue
            m, t = load_model(path, base_model_name=model_name)
            rows.extend(evaluate_all(m, t, label, model_name))
            del m; torch.cuda.empty_cache()
    return rows


def eval_merged(model_name):
    """Find and evaluate all merged models for a given base model."""
    rows = []
    for calib_name, calib_spec in CALIB_DATASETS.items():
        for ratio in COMPRESSION_RATIOS:
            path = merged_model_path(model_name, calib_spec, ratio)
            label = f"hc_smoe|{calib_name}|r={ratio}"
            print(f"\n--- {label} ---")
            print(f"    Path: {path}")
            if not glob.glob(f"{path}/*.safetensors") and not glob.glob(f"{path}/*.bin"):
                print(f"    SKIP: no model files found")
                continue
            m, t = load_model(path, base_model_name=model_name)
            rows.extend(evaluate_all(m, t, label, model_name))
            del m; torch.cuda.empty_cache()
    return rows


---
## Model 1: Qwen/Qwen1.5-MoE-A2.7B-Chat

### Baseline

In [4]:
m1_name = MODELS[0]
print(f"Loading {m1_name}")
m1, t1 = load_model(m1_name)
results.extend(evaluate_all(m1, t1, "baseline", m1_name))
del m1; torch.cuda.empty_cache()

Loading Qwen/Qwen1.5-MoE-A2.7B-Chat


Loading checkpoint shards: 100%|██████████| 8/8 [03:03<00:00, 22.95s/it]



  === baseline ===
    MMLU (1000 samples): 0.5350
    CommonsenseQA (1000 samples): 0.7580


Generating test split: 100%|██████████| 500/500 [00:00<00:00, 194216.71 examples/s]


    OpenBookQA (1000 samples): 0.6860


Generating validation split: 100%|██████████| 1267/1267 [00:00<00:00, 469533.77 examples/s]


    Winogrande (1000 samples): 0.5460

  --- Summary (baseline) ---
    MMLU                 0.5350
    CommonsenseQA        0.7580
    OpenBookQA           0.6860
    Winogrande           0.5460



### REAP Pruning — commands

In [5]:
m1_name = MODELS[0]
CALIB_DATASETS = {
    "mmlu_train":           "cais/mmlu[all](auxiliary_train):1024",
    "commonsense_qa_train": "tau/commonsense_qa(train):1024",
    "openbookqa_train":     "allenai/openbookqa[main](train):1024",
    "winogrande_train":     "allenai/winogrande[winogrande_debiased](train):1024",
}

COMPRESSION_RATIOS = [0.05, 0.1]

print_reap_cmds(m1_name)

# REAP pruning — run in Docker terminal:

# mmlu_train, ratio=0.05
CUDA_VISIBLE_DEVICES=0,1 bash experiments/pruning-cli.sh 0,1 Qwen/Qwen1.5-MoE-A2.7B-Chat reap 42 0.05 "cais/mmlu[all](auxiliary_train):1024" false false false false false

# mmlu_train, ratio=0.1
CUDA_VISIBLE_DEVICES=0,1 bash experiments/pruning-cli.sh 0,1 Qwen/Qwen1.5-MoE-A2.7B-Chat reap 42 0.1 "cais/mmlu[all](auxiliary_train):1024" false false false false false

# commonsense_qa_train, ratio=0.05
CUDA_VISIBLE_DEVICES=0,1 bash experiments/pruning-cli.sh 0,1 Qwen/Qwen1.5-MoE-A2.7B-Chat reap 42 0.05 "tau/commonsense_qa(train):1024" false false false false false

# commonsense_qa_train, ratio=0.1
CUDA_VISIBLE_DEVICES=0,1 bash experiments/pruning-cli.sh 0,1 Qwen/Qwen1.5-MoE-A2.7B-Chat reap 42 0.1 "tau/commonsense_qa(train):1024" false false false false false

# openbookqa_train, ratio=0.05
CUDA_VISIBLE_DEVICES=0,1 bash experiments/pruning-cli.sh 0,1 Qwen/Qwen1.5-MoE-A2.7B-Chat reap 42 0.05 "allenai/openbookqa[main](train):

### REAP Pruning — eval compressed

In [4]:
m1_name = MODELS[0]
CALIB_DATASETS = {
    "mmlu_train":           "cais/mmlu[all](auxiliary_train):1024",
    "commonsense_qa_train": "tau/commonsense_qa(train):1024",
    "openbookqa_train":     "allenai/openbookqa[main](train):1024",
    "winogrande_train":     "allenai/winogrande[winogrande_debiased](train):1024",
}

COMPRESSION_RATIOS = [0.05, 0.1]
results.extend(eval_pruned(m1_name))


--- reap|mmlu_train|r=0.05 ---
    Path: artifacts/Qwen1.5-MoE-A2.7B-Chat/mmlu_all__auxiliary_train__1024/pruned_models/reap-renorm_true-seed_42-0.05


Loading checkpoint shards: 100%|██████████| 6/6 [00:07<00:00,  1.25s/it]



  === reap|mmlu_train|r=0.05 ===
    MMLU (1000 samples): 0.5140
    CommonsenseQA (1000 samples): 0.7580
    OpenBookQA (1000 samples): 0.6780
    Winogrande (1000 samples): 0.5390

  --- Summary (reap|mmlu_train|r=0.05) ---
    MMLU                 0.5140
    CommonsenseQA        0.7580
    OpenBookQA           0.6780
    Winogrande           0.5390


--- reap|mmlu_train|r=0.1 ---
    Path: artifacts/Qwen1.5-MoE-A2.7B-Chat/mmlu_all__auxiliary_train__1024/pruned_models/reap-renorm_true-seed_42-0.10


Loading checkpoint shards: 100%|██████████| 6/6 [00:05<00:00,  1.11it/s]



  === reap|mmlu_train|r=0.1 ===
    MMLU (1000 samples): 0.5110
    CommonsenseQA (1000 samples): 0.7420
    OpenBookQA (1000 samples): 0.6720
    Winogrande (1000 samples): 0.5460

  --- Summary (reap|mmlu_train|r=0.1) ---
    MMLU                 0.5110
    CommonsenseQA        0.7420
    OpenBookQA           0.6720
    Winogrande           0.5460


--- reap|commonsense_qa_train|r=0.05 ---
    Path: artifacts/Qwen1.5-MoE-A2.7B-Chat/commonsense_qa_train__1024/pruned_models/reap-renorm_true-seed_42-0.05


Loading checkpoint shards: 100%|██████████| 6/6 [00:12<00:00,  2.15s/it]


... (7 outputs left out)

Loading checkpoint shards: 100%|██████████| 6/6 [00:46<00:00,  7.80s/it]



  === reap|winogrande_train|r=0.05 ===
    MMLU (1000 samples): 0.5310
    CommonsenseQA (1000 samples): 0.7530
    OpenBookQA (1000 samples): 0.6800
    Winogrande (1000 samples): 0.5360

  --- Summary (reap|winogrande_train|r=0.05) ---
    MMLU                 0.5310
    CommonsenseQA        0.7530
    OpenBookQA           0.6800
    Winogrande           0.5360


--- reap|winogrande_train|r=0.1 ---
    Path: artifacts/Qwen1.5-MoE-A2.7B-Chat/winogrande_winogrande_debiased__train__1024/pruned_models/reap-renorm_true-seed_42-0.10


Loading checkpoint shards: 100%|██████████| 6/6 [00:05<00:00,  1.20it/s]



  === reap|winogrande_train|r=0.1 ===
    MMLU (1000 samples): 0.5040
    CommonsenseQA (1000 samples): 0.7430
    OpenBookQA (1000 samples): 0.5960
    Winogrande (1000 samples): 0.5310

  --- Summary (reap|winogrande_train|r=0.1) ---
    MMLU                 0.5040
    CommonsenseQA        0.7430
    OpenBookQA           0.5960
    Winogrande           0.5310



### HC-SMoE Merging — commands

In [6]:
m1_name = MODELS[0]
CALIB_DATASETS = {
    "mmlu_train":           "cais/mmlu[all](auxiliary_train):128",
    "commonsense_qa_train": "tau/commonsense_qa(train):128",
    "openbookqa_train":     "allenai/openbookqa[main](train):128",
    "winogrande_train":     "allenai/winogrande[winogrande_debiased](train):128",
}

COMPRESSION_RATIOS = [0.05, 0.1]

print_hc_smoe_cmds(m1_name)

# HC-SMoE merging — run in Docker terminal:

# mmlu_train, ratio=0.05
CUDA_VISIBLE_DEVICES=0,1 bash experiments/merging-cli.sh 0,1 Qwen/Qwen1.5-MoE-A2.7B-Chat hc_smoe 42 0.05 "cais/mmlu[all](auxiliary_train):128" false false false false false

# mmlu_train, ratio=0.1
CUDA_VISIBLE_DEVICES=0,1 bash experiments/merging-cli.sh 0,1 Qwen/Qwen1.5-MoE-A2.7B-Chat hc_smoe 42 0.1 "cais/mmlu[all](auxiliary_train):128" false false false false false

# commonsense_qa_train, ratio=0.05
CUDA_VISIBLE_DEVICES=0,1 bash experiments/merging-cli.sh 0,1 Qwen/Qwen1.5-MoE-A2.7B-Chat hc_smoe 42 0.05 "tau/commonsense_qa(train):128" false false false false false

# commonsense_qa_train, ratio=0.1
CUDA_VISIBLE_DEVICES=0,1 bash experiments/merging-cli.sh 0,1 Qwen/Qwen1.5-MoE-A2.7B-Chat hc_smoe 42 0.1 "tau/commonsense_qa(train):128" false false false false false

# openbookqa_train, ratio=0.05
CUDA_VISIBLE_DEVICES=0,1 bash experiments/merging-cli.sh 0,1 Qwen/Qwen1.5-MoE-A2.7B-Chat hc_smoe 42 0.05 "allenai/openbookqa

### HC-SMoE Merging — eval compressed

In [4]:
m1_name = MODELS[0]
CALIB_DATASETS = {
    "mmlu_train":           "cais/mmlu[all](auxiliary_train):128",
    "commonsense_qa_train": "tau/commonsense_qa(train):128",
    "openbookqa_train":     "allenai/openbookqa[main](train):128",
    "winogrande_train":     "allenai/winogrande[winogrande_debiased](train):128",
}

COMPRESSION_RATIOS = [0.05, 0.1]
results.extend(eval_merged(m1_name))


--- hc_smoe|mmlu_train|r=0.05 ---
    Path: artifacts/Qwen1.5-MoE-A2.7B-Chat/mmlu_all__auxiliary_train__128/merged_models/hc_smoe-seed_42_0.05/hc_smoe


Loading checkpoint shards: 100%|██████████| 6/6 [01:26<00:00, 14.46s/it]



  === hc_smoe|mmlu_train|r=0.05 ===
    MMLU (1000 samples): 0.5250
    CommonsenseQA (1000 samples): 0.7510
    OpenBookQA (1000 samples): 0.6820
    Winogrande (1000 samples): 0.5490

  --- Summary (hc_smoe|mmlu_train|r=0.05) ---
    MMLU                 0.5250
    CommonsenseQA        0.7510
    OpenBookQA           0.6820
    Winogrande           0.5490


--- hc_smoe|mmlu_train|r=0.1 ---
    Path: artifacts/Qwen1.5-MoE-A2.7B-Chat/mmlu_all__auxiliary_train__128/merged_models/hc_smoe-seed_42_0.1/hc_smoe


Loading checkpoint shards: 100%|██████████| 6/6 [00:06<00:00,  1.08s/it]



  === hc_smoe|mmlu_train|r=0.1 ===
    MMLU (1000 samples): 0.5020
    CommonsenseQA (1000 samples): 0.7270
    OpenBookQA (1000 samples): 0.6440
    Winogrande (1000 samples): 0.5390

  --- Summary (hc_smoe|mmlu_train|r=0.1) ---
    MMLU                 0.5020
    CommonsenseQA        0.7270
    OpenBookQA           0.6440
    Winogrande           0.5390


--- hc_smoe|commonsense_qa_train|r=0.05 ---
    Path: artifacts/Qwen1.5-MoE-A2.7B-Chat/commonsense_qa_train__128/merged_models/hc_smoe-seed_42_0.05/hc_smoe


Loading checkpoint shards: 100%|██████████| 6/6 [00:08<00:00,  1.49s/it]


... (7 outputs left out)

Loading checkpoint shards: 100%|██████████| 6/6 [05:39<00:00, 56.61s/it]



  === hc_smoe|winogrande_train|r=0.05 ===
    MMLU (1000 samples): 0.5170
    CommonsenseQA (1000 samples): 0.7540
    OpenBookQA (1000 samples): 0.6760
    Winogrande (1000 samples): 0.5430

  --- Summary (hc_smoe|winogrande_train|r=0.05) ---
    MMLU                 0.5170
    CommonsenseQA        0.7540
    OpenBookQA           0.6760
    Winogrande           0.5430


--- hc_smoe|winogrande_train|r=0.1 ---
    Path: artifacts/Qwen1.5-MoE-A2.7B-Chat/winogrande_winogrande_debiased__train__128/merged_models/hc_smoe-seed_42_0.1/hc_smoe


Loading checkpoint shards: 100%|██████████| 6/6 [05:55<00:00, 59.19s/it]



  === hc_smoe|winogrande_train|r=0.1 ===
    MMLU (1000 samples): 0.5110
    CommonsenseQA (1000 samples): 0.7480
    OpenBookQA (1000 samples): 0.6200
    Winogrande (1000 samples): 0.5530

  --- Summary (hc_smoe|winogrande_train|r=0.1) ---
    MMLU                 0.5110
    CommonsenseQA        0.7480
    OpenBookQA           0.6200
    Winogrande           0.5530



---
## Model 3: microsoft/Phi-mini-MoE-instruct

### Baseline

In [4]:
m3_name = MODELS[2]
print(f"Loading {m3_name}")
m3, t3 = load_model(m3_name)
results.extend(evaluate_all(m3, t3, "baseline", m3_name))
del m3; torch.cuda.empty_cache()

Loading microsoft/Phi-mini-MoE-instruct


Loading checkpoint shards: 100%|██████████| 4/4 [00:19<00:00,  4.92s/it]



  === baseline ===
    MMLU (1000 samples): 0.6670
    CommonsenseQA (1000 samples): 0.7530
    OpenBookQA (1000 samples): 0.8480
    Winogrande (1000 samples): 0.7420

  --- Summary (baseline) ---
    MMLU                 0.6670
    CommonsenseQA        0.7530
    OpenBookQA           0.8480
    Winogrande           0.7420



### REAP Pruning — commands

In [8]:
print_reap_cmds(m3_name)

# REAP pruning — run in Docker terminal:

# mmlu_train, ratio=0.1
CUDA_VISIBLE_DEVICES=0,1 bash experiments/pruning-cli.sh 0,1 microsoft/Phi-mini-MoE-instruct reap 42 0.1 "cais/mmlu[all](auxiliary_train):15000" false false false false false

# mmlu_train, ratio=0.25
CUDA_VISIBLE_DEVICES=0,1 bash experiments/pruning-cli.sh 0,1 microsoft/Phi-mini-MoE-instruct reap 42 0.25 "cais/mmlu[all](auxiliary_train):15000" false false false false false


### REAP Pruning — eval compressed

In [4]:
m3_name = MODELS[2]
CALIB_DATASETS = {
    "mmlu_train":           "cais/mmlu[all](auxiliary_train):1024",
    "commonsense_qa_train": "tau/commonsense_qa(train):1024",
    "openbookqa_train":     "allenai/openbookqa[main](train):1024",
    "winogrande_train":     "allenai/winogrande[winogrande_debiased](train):1024",
}

COMPRESSION_RATIOS = [0.06, 0.12]
results.extend(eval_pruned(m3_name))


--- reap|mmlu_train|r=0.06 ---
    Path: artifacts/Phi-mini-MoE-instruct/mmlu_all__auxiliary_train__1024/pruned_models/reap-renorm_true-seed_42-0.06


Loading checkpoint shards: 100%|██████████| 3/3 [01:33<00:00, 31.10s/it]



  === reap|mmlu_train|r=0.06 ===
    MMLU (1000 samples): 0.6560
    CommonsenseQA (1000 samples): 0.7530
    OpenBookQA (1000 samples): 0.8220
    Winogrande (1000 samples): 0.7440

  --- Summary (reap|mmlu_train|r=0.06) ---
    MMLU                 0.6560
    CommonsenseQA        0.7530
    OpenBookQA           0.8220
    Winogrande           0.7440


--- reap|mmlu_train|r=0.12 ---
    Path: artifacts/Phi-mini-MoE-instruct/mmlu_all__auxiliary_train__1024/pruned_models/reap-renorm_true-seed_42-0.12


Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.32it/s]



  === reap|mmlu_train|r=0.12 ===
    MMLU (1000 samples): 0.6370
    CommonsenseQA (1000 samples): 0.7560
    OpenBookQA (1000 samples): 0.8200
    Winogrande (1000 samples): 0.7310

  --- Summary (reap|mmlu_train|r=0.12) ---
    MMLU                 0.6370
    CommonsenseQA        0.7560
    OpenBookQA           0.8200
    Winogrande           0.7310


--- reap|commonsense_qa_train|r=0.06 ---
    Path: artifacts/Phi-mini-MoE-instruct/commonsense_qa_train__1024/pruned_models/reap-renorm_true-seed_42-0.06


Loading checkpoint shards: 100%|██████████| 3/3 [00:23<00:00,  7.76s/it]


... (7 outputs left out)

Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.06s/it]



  === reap|winogrande_train|r=0.06 ===
    MMLU (1000 samples): 0.6510
    CommonsenseQA (1000 samples): 0.7580
    OpenBookQA (1000 samples): 0.8340
    Winogrande (1000 samples): 0.7410

  --- Summary (reap|winogrande_train|r=0.06) ---
    MMLU                 0.6510
    CommonsenseQA        0.7580
    OpenBookQA           0.8340
    Winogrande           0.7410


--- reap|winogrande_train|r=0.12 ---
    Path: artifacts/Phi-mini-MoE-instruct/winogrande_winogrande_debiased__train__1024/pruned_models/reap-renorm_true-seed_42-0.12


Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.31it/s]



  === reap|winogrande_train|r=0.12 ===
    MMLU (1000 samples): 0.6020
    CommonsenseQA (1000 samples): 0.7440
    OpenBookQA (1000 samples): 0.8220
    Winogrande (1000 samples): 0.7380

  --- Summary (reap|winogrande_train|r=0.12) ---
    MMLU                 0.6020
    CommonsenseQA        0.7440
    OpenBookQA           0.8220
    Winogrande           0.7380



### HC-SMoE Merging — commands

In [7]:
CALIB_DATASETS = {
    "mmlu_train": "cais/mmlu[all](auxiliary_train):128",
    "commonsense_qa_train": "tau/commonsense_qa(train):128",
    #"gsm8k_train": "openai/gsm8k[main](train):8",
}
print_hc_smoe_cmds(m3_name)

# HC-SMoE merging — run in Docker terminal:

# mmlu_train, ratio=0.05
CUDA_VISIBLE_DEVICES=0,1 bash experiments/merging-cli.sh 0,1 microsoft/Phi-mini-MoE-instruct hc_smoe 42 0.05 "cais/mmlu[all](auxiliary_train):128" false false false false false

# mmlu_train, ratio=0.1
CUDA_VISIBLE_DEVICES=0,1 bash experiments/merging-cli.sh 0,1 microsoft/Phi-mini-MoE-instruct hc_smoe 42 0.1 "cais/mmlu[all](auxiliary_train):128" false false false false false

# commonsense_qa_train, ratio=0.05
CUDA_VISIBLE_DEVICES=0,1 bash experiments/merging-cli.sh 0,1 microsoft/Phi-mini-MoE-instruct hc_smoe 42 0.05 "tau/commonsense_qa(train):128" false false false false false

# commonsense_qa_train, ratio=0.1
CUDA_VISIBLE_DEVICES=0,1 bash experiments/merging-cli.sh 0,1 microsoft/Phi-mini-MoE-instruct hc_smoe 42 0.1 "tau/commonsense_qa(train):128" false false false false false


### HC-SMoE Merging — eval compressed

In [4]:
CALIB_DATASETS = {
    "mmlu_train":           "cais/mmlu[all](auxiliary_train):128",
    "commonsense_qa_train": "tau/commonsense_qa(train):128",
    "openbookqa_train":     "allenai/openbookqa[main](train):128",
    "winogrande_train":     "allenai/winogrande[winogrande_debiased](train):128",
}

COMPRESSION_RATIOS = [0.05, 0.1]
m3_name = MODELS[2]
results.extend(eval_merged(m3_name))


--- hc_smoe|mmlu_train|r=0.05 ---
    Path: artifacts/Phi-mini-MoE-instruct/mmlu_all__auxiliary_train__128/merged_models/hc_smoe-seed_42_0.05/hc_smoe


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.47it/s]



  === hc_smoe|mmlu_train|r=0.05 ===
    MMLU (1000 samples): 0.6450
    CommonsenseQA (1000 samples): 0.7470
    OpenBookQA (1000 samples): 0.8000
    Winogrande (1000 samples): 0.7230

  --- Summary (hc_smoe|mmlu_train|r=0.05) ---
    MMLU                 0.6450
    CommonsenseQA        0.7470
    OpenBookQA           0.8000
    Winogrande           0.7230


--- hc_smoe|mmlu_train|r=0.1 ---
    Path: artifacts/Phi-mini-MoE-instruct/mmlu_all__auxiliary_train__128/merged_models/hc_smoe-seed_42_0.1/hc_smoe


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.64it/s]



  === hc_smoe|mmlu_train|r=0.1 ===
    MMLU (1000 samples): 0.5960
    CommonsenseQA (1000 samples): 0.7360
    OpenBookQA (1000 samples): 0.7820
    Winogrande (1000 samples): 0.6980

  --- Summary (hc_smoe|mmlu_train|r=0.1) ---
    MMLU                 0.5960
    CommonsenseQA        0.7360
    OpenBookQA           0.7820
    Winogrande           0.6980


--- hc_smoe|commonsense_qa_train|r=0.05 ---
    Path: artifacts/Phi-mini-MoE-instruct/commonsense_qa_train__128/merged_models/hc_smoe-seed_42_0.05/hc_smoe


Loading checkpoint shards: 100%|██████████| 4/4 [02:12<00:00, 33.16s/it]


... (7 outputs left out)

Loading checkpoint shards: 100%|██████████| 4/4 [00:22<00:00,  5.74s/it]



  === hc_smoe|winogrande_train|r=0.05 ===
    MMLU (1000 samples): 0.6360
    CommonsenseQA (1000 samples): 0.7380
    OpenBookQA (1000 samples): 0.8220
    Winogrande (1000 samples): 0.7390

  --- Summary (hc_smoe|winogrande_train|r=0.05) ---
    MMLU                 0.6360
    CommonsenseQA        0.7380
    OpenBookQA           0.8220
    Winogrande           0.7390


--- hc_smoe|winogrande_train|r=0.1 ---
    Path: artifacts/Phi-mini-MoE-instruct/winogrande_winogrande_debiased__train__128/merged_models/hc_smoe-seed_42_0.1/hc_smoe


Loading checkpoint shards: 100%|██████████| 4/4 [01:23<00:00, 20.81s/it]



  === hc_smoe|winogrande_train|r=0.1 ===
    MMLU (1000 samples): 0.6020
    CommonsenseQA (1000 samples): 0.7310
    OpenBookQA (1000 samples): 0.7680
    Winogrande (1000 samples): 0.7110

  --- Summary (hc_smoe|winogrande_train|r=0.1) ---
    MMLU                 0.6020
    CommonsenseQA        0.7310
    OpenBookQA           0.7680
    Winogrande           0.7110



---
## Results

In [ ]:
df = pd.DataFrame(results)
df.to_csv("baselines_results.csv", index=False)
print(df.pivot_table(index=["model", "variant"], columns="benchmark", values="accuracy").to_string())

In [ ]:
for name in MODELS:
    sub = df[df["model"] == name]
    print(f"\n{name}:")
    print(sub.pivot_table(index="variant", columns="benchmark", values="accuracy").to_string())
    print()